# Category-filtered object RAG: design and plan

Question + category → mapped active documents → eligible table/outer-row vectors → cosine ranking → exact cited evidence → Ollama code answer.

Reuse the existing index without ingestion or checkpoint changes. Match the combined and V4.0/V5.0 workbook mappings through explicit corpus-root aliases, retain corpus and revision labels, and report missing coverage. Traverse row-group wrappers for outer rows; exclude cells and deeper rows from vector candidates. Validate identity, embedding configuration and dimensions; query eligible IDs before ranking.

Implementation: configure paths and services; load categories/mappings; open the catalog read-only and existing collection; inspect coverage; retrieve and inspect exact evidence; generate version-specific fenced snippets with citations; close resources. Defaults: supplied index, top-k 8, 24,000 context characters, Ollama RAG model falling back to the summary model. Complete blocks exceeding the budget are reported as omitted. Unsupported coding requests must receive an insufficient-evidence answer.

Validation: offline fake-provider and ephemeral-index tests, notebook validation and cell compilation, and read-only real catalog coverage inspection. No corpus rebuilds, summary requests or checkpoint resets.


Every answer includes a separate source-attributed summary of the retrieved objects, regardless of whether the coding request is supported.

## Application configuration

Reusable retrieval logic lives in [01.rag.object.ipynb](01.rag.object.ipynb), exported as `ribosome.rag.object`. Configuration and client lifecycle stay here.

In [ ]:
# | notest
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from ribosome.rag.object import ObjectRAG

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").exists() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
load_dotenv(PROJ_ROOT / ".env", override=False)
# INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261022-152558-151254"))
INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261026-090625-217683"))
if not INDEX_DIR.is_absolute(): INDEX_DIR = PROJ_ROOT / INDEX_DIR
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")
HOSTS = sorted(set(x.strip().rstrip("/") for x in os.getenv("OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434").split(",") if x.strip()))
RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
RAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")
MAX_CONTEXT_CHARS = 24000
REQUEST_TIMEOUT = 300
MAX_OUTPUT_TOKENS = 2048



## Open existing index and services
Run explicitly; no embedding or generation occurs in this cell.

In [ ]:
# | notest
import chromadb
from ollama import AsyncClient
if not (INDEX_DIR / 'catalog.sqlite3').is_file() or not (INDEX_DIR / 'chroma').is_dir():
    raise FileNotFoundError(INDEX_DIR)
headers = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
embedding_client = AsyncClient(host=HOSTS[0], headers=headers, timeout=REQUEST_TIMEOUT)
generation_client = AsyncClient(host=RAG_HOST, headers=headers, timeout=REQUEST_TIMEOUT)
collection = chromadb.PersistentClient(path=str(INDEX_DIR/'chroma')).get_collection('object_summaries', embedding_function=None)
app = ObjectRAG(
    INDEX_DIR, collection, embedding_client, generation_client,
    mapping_dir=PROJ_ROOT / 'assets/coding_categories',
    embed_model=EMBED_MODEL,
    embedding_config=dict(provider='ollama-pool', hosts=HOSTS),
    rag_model=RAG_MODEL, max_context_chars=MAX_CONTEXT_CHARS,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
async def retrieve(question, category, *, top_k=8):
    return await app.retrieve(question, category, top_k=top_k)
async def answer(question, category, *, top_k=8):
    return await app.answer(question, category, top_k=top_k)


## Category coverage

This suite has one prompt for each of the **17 application categories other than
`general_operation`**. Questions may use either mapped manual revision, with the
answer identifying its source. The dedicated **General-operation manual version
tests** section at the bottom covers the B-5 and C-1 instructions and alarm codes.

Set `COVERAGE_CASE_IDS` to selected category case IDs, or leave it `None` for all
17 prompts. The following two sections retrieve evidence and generate answers for
this suite only. Defining or displaying prompts makes no model requests. Shared
helpers also serve the separate manual suite, which has its own selection and
result dictionaries.


In [ ]:
# | notest
COVERAGE_CASES = [{'id': 'category_adhesive_dispensing',
  'category': 'adhesive_dispensing',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何按涂胶手册配置胶机并编写一次涂胶流程，包含开始出胶、沿轨迹涂胶和停止出胶？请说明工艺文件和IO的准备条件。',
  'references': []},
 {'id': 'category_arc_welding',
  'category': 'arc_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何使用ARCON开始弧焊、沿焊缝运动并用ARCOFF结束焊接？请说明起弧文件与焊接文件的区别。',
  'references': []},
 {'id': 'category_fds',
  'category': 'fds',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何按FDS手册配置主机工作参数和IO，并组织一次FDS作业的启动、完成状态确认及后续动作？',
  'references': []},
 {'id': 'category_grinding',
  'category': 'grinding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何首次配置打磨力控制器并完成标定，再设置打磨力、位移及出入刀监控，组织一次打磨作业？',
  'references': []},
 {'id': 'category_laser_welding',
  'category': 'laser_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '激光焊作业需要哪些激光器、通讯和IO配置？如何按手册组织激光开启、焊接运动和激光关闭的程序？',
  'references': []},
 {'id': 'category_load_analysis',
  'category': 'load_analysis',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '使用负载分析软件时，如何确认工具数模坐标系与机器人法兰坐标系一致，并查看和解释负载分析结果？只需说明操作流程，不要虚构控制指令。',
  'references': []},
 {'id': 'category_material_handling',
  'category': 'material_handling',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置搬运应用的抓手、工件传感器和电磁阀，并组织抓取工件、确认抓取状态、搬运后释放的作业？',
  'references': []},
 {'id': 'category_offline_interface',
  'category': 'offline_interface',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '使用新松机器人离线接口库时，上位机初始化、建立网络连接、发送运动或下载作业、断开并释放资源的正确调用顺序是什么？请选择一种手册支持的语言举例。',
  'references': []},
 {'id': 'category_palletizing',
  'category': 'palletizing',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何从开启码垛应用、PLC通讯准备、工具与用户坐标系设置开始，配置产品和码垛工艺，再组织一次码垛作业？',
  'references': []},
 {'id': 'category_pneumatic_spot_welding',
  'category': 'pneumatic_spot_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置气动点焊的设备交互IO、焊接压力和焊点参数，并组织加压、焊接、完成确认及松开焊钳的流程？',
  'references': []},
 {'id': 'category_spot_welding',
  'category': 'spot_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '伺服点焊作业前如何完成焊钳轴校零、压力标定和焊点参数配置，并组织一次加压、焊接及打开焊钳的流程？',
  'references': []},
 {'id': 'category_spr_riveting',
  'category': 'spr_riveting',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置SPR铆机的工作模式、IO及通讯，并按基本作业流程完成一次铆接和防漏铆确认？',
  'references': []},
 {'id': 'category_stud_welding',
  'category': 'stud_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置螺柱焊焊机的工作参数、IO及通讯，并按基本作业流程完成一次焊接及防漏焊确认？',
  'references': []},
 {'id': 'category_teaching_free_welding',
  'category': 'teaching_free_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '免示教焊接软件如何配置机器人与视觉设备、创建工件、设置扫描区域和焊缝规则，再规划并执行焊接任务？请区分服务端配置和客户端操作。',
  'references': []},
 {'id': 'category_tool_quick_change',
  'category': 'tool_quick_change',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置快换停靠站、快换盘和IO，并组织放下当前工具、确认解锁、拾取目标工具和确认锁紧的基本作业流程？',
  'references': []},
 {'id': 'category_tracking',
  'category': 'tracking',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何使用FL_GETV读取第一路传送带的实时速度，并在跟随作业中使用该速度？请说明R变量、通路参数及单位。',
  'references': []},
 {'id': 'category_vision',
  'category': 'vision',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何给相机发送拍照指令、获取测量结果并计算位姿偏移？请按同一手册版本说明视觉作业参数及指令调用顺序。',
  'references': []}]


In [ ]:
# | notest
import html
import re
from copy import deepcopy

def coverage_plain(text):
    return ' '.join(html.unescape(re.sub(r'<[^>]*>', ' ', text)).split())

def coverage_case_metrics(case, result, error=''):
    hits = result.get('hits', [])
    # Version-specific cases must find their own corpus and manual, even though
    # production retrieval is category-filtered and may return both revisions.
    matching = [hit for hit in hits if not case['profile'] or (
        hit['manual']['profile'] == case['profile'] and
        Path(hit['record']['source_file']).stem == Path(case['references'][0]['docx']).stem)]
    headings = [heading.split(maxsplit=1)[0] for hit in matching
                for heading in hit['record'].get('section_headings', []) if heading.strip()]
    source_text = '\n'.join(coverage_plain(hit['record']['content']) for hit in matching)
    anchors_found = all(re.search(r'(?<![A-Za-z0-9_])'+re.escape(term)+r'(?![A-Za-z0-9_])',
                                 source_text, flags=re.IGNORECASE) for term in case['anchors'])
    return dict(case_id=case['id'], category=case['category'], kind=case['kind'],
                expected_revision=case['revision'], expected_section=case['section'],
                hits=len(hits), expected_manual_hit=bool(matching) if case['profile'] else None,
                expected_section_hit=case['section'] in headings if case['section'] else None,
                anchors_in_matching_hits=anchors_found if case['anchors'] else None,
                context_chars=len(result.get('context', '')), omissions=len(result.get('omissions', [])),
                missing_manuals=len(result.get('coverage', {}).get('missing', [])), error=error)

def show_coverage_retrieval(case_id, *, cases=None, results=None, errors=None):
    cases = coverage_cases if cases is None else cases
    results = coverage_retrieval_results if results is None else results
    errors = coverage_retrieval_errors if errors is None else errors
    case = next(case for case in cases if case['id'] == case_id)
    display(Markdown(f"### {case_id}\n\n{case['question']}"))
    if case_id in errors:
        print(errors[case_id])
        return
    result = results[case_id]
    display(pd.DataFrame([dict(distance=hit['distance'], manual=hit['manual'],
                               lines=(hit['record']['attributes'].get('line_start'),
                                      hit['record']['attributes'].get('line_end')),
                               summary=hit['record'].get('summary', ''),
                               source=hit['record']['source_file']) for hit in result['hits']]))
    print(result['context'])
    print('Omitted:', result['omissions'])
    # Reference excerpts remain reviewer data and are never added to the prompt.
    if case['references']:
        display(pd.DataFrame(case['references']))

class CoverageAnswerReplay(ObjectRAG):
    """Run the production answer method on the exact retrieved test evidence."""
    def __init__(self, application, evidence, *, top_k=8):
        self.generator = application.generator
        self.rag_model = application.rag_model
        self.max_output_tokens = application.max_output_tokens
        self.evidence = evidence
        self.top_k = top_k
        self.last_result = None

    async def retrieve(self, question, category, *, top_k=8):
        if (question, category, top_k) != (
            self.evidence['question'], self.evidence['category'], self.top_k
        ):
            raise ValueError('Coverage replay requires the same question, category and top-k')
        self.last_result = deepcopy(self.evidence)
        return self.last_result

async def run_coverage_retrievals(application, cases, *, top_k, results, errors):
    """Retrieve into this suite's dictionaries; retain completed cases on interruption."""
    for number, case in enumerate(cases, 1):
        print(f"[{number}/{len(cases)}] Retrieve {case['id']}", flush=True)
        try:
            results[case['id']] = await application.retrieve(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case['id']] = f'{type(error).__name__}: {error}'
    return pd.DataFrame([
        coverage_case_metrics(case, results.get(case['id'], {}),
                              errors.get(case['id'], ''))
        for case in cases])

async def run_coverage_answers(application, cases, *, top_k, retrieval_results,
                               retrieval_errors, results, errors):
    """Answer from this suite's frozen evidence; never make another retrieval request."""
    for number, case in enumerate(cases, 1):
        case_id = case['id']
        display(Markdown(f"## [{number}/{len(cases)}] {case_id}\n\n{case['question']}"))
        if case_id not in retrieval_results:
            errors[case_id] = retrieval_errors.get(case_id, 'Retrieve this case first')
            display(Markdown('Retrieval failed: ' + errors[case_id]))
            continue
        replay = CoverageAnswerReplay(application, retrieval_results[case_id], top_k=top_k)
        try:
            results[case_id] = await replay.answer(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case_id] = f'{type(error).__name__}: {error}'
            results[case_id] = replay.last_result or {}
        finally:
            if case_id not in results and replay.last_result is not None:
                results[case_id] = replay.last_result
                errors[case_id] = 'Generation interrupted before completion'
            case_result = results.get(case_id, {})
            if 'retrieved_summary' in case_result:
                display(Markdown('### Retrieved content summary\n\n' + case_result['retrieved_summary']))
        if case_id in errors:
            display(Markdown('Generation failed: ' + errors[case_id]))
        else:
            display(Markdown('### Coding or diagnostic answer\n\n' + case_result['answer']))
    return pd.DataFrame([
        dict(**coverage_case_metrics(case, results.get(case['id'], {}),
                                     errors.get(case['id'], '')),
             example_status=results.get(case['id'], {}).get('example_status', 'not generated'))
        for case in cases])


In [ ]:
# | notest
display(pd.DataFrame([(category, name) for category, name in app.categories.items()
                      if category != 'general_operation'], columns=['category', 'chinese_name']))
assert {case['category'] for case in COVERAGE_CASES} == set(app.categories) - {'general_operation'}
COVERAGE_TOP_K = 8
COVERAGE_CASE_IDS = None  # Example: ['category_arc_welding', 'category_vision']
known_case_ids = {case['id'] for case in COVERAGE_CASES}
if COVERAGE_CASE_IDS is not None:
    if not COVERAGE_CASE_IDS or set(COVERAGE_CASE_IDS) - known_case_ids:
        raise ValueError('Select at least one known coverage case ID')
coverage_cases = [case for case in COVERAGE_CASES
                  if COVERAGE_CASE_IDS is None or case['id'] in COVERAGE_CASE_IDS]
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(pd.DataFrame(coverage_cases)[
        ['id', 'category', 'question']])
category_coverage_rows = []
for category in dict.fromkeys(case['category'] for case in coverage_cases):
    mapped_coverage = app.snapshot(category)[2]
    category_coverage_rows.append(dict(category=category, indexed=mapped_coverage['indexed'],
                                       missing=mapped_coverage['missing']))
display(pd.DataFrame(category_coverage_rows))


In [ ]:
# | notest
INSPECT_CASE_ID = coverage_cases[0]['id']  # Change to any selected ID to inspect its exact evidence.


## Retrieve and inspect exact evidence

Run the selected category prompts through the existing category-filtered retriever.
Results are retained by case ID in `coverage_retrieval_results`, with per-case
failures in `coverage_retrieval_errors`; completed results remain inspectable after
interruption. Use `show_coverage_retrieval(case_id)` for any selected category case
to inspect the matched objects, raw context and omissions.

The category table reports hit count, context size, omitted objects, missing mapped
manuals and errors. These are retrieval diagnostics, not answer-correctness scores.
Version, subsection and alarm-code checks belong to the dedicated manual suite
below. The original production candidate restrictions still apply.


In [ ]:
# | notest
coverage_retrieval_results, coverage_retrieval_errors = {}, {}
coverage_retrieval_metrics = await run_coverage_retrievals(
    app, coverage_cases, top_k=COVERAGE_TOP_K,
    results=coverage_retrieval_results, errors=coverage_retrieval_errors)
display(coverage_retrieval_metrics[
    ['case_id', 'category', 'hits', 'context_chars', 'omissions', 'missing_manuals', 'error']])
show_coverage_retrieval(INSPECT_CASE_ID)


## Retrieved content summary and coding answer

Always display a source-attributed summary of the found objects, even when their content cannot support the requested code. The summary uses stored object summaries and remains available if generation fails. With no hits, explicitly report that no relevant indexed content was found.

The coding answer ends with a meaningful fenced code snippet for the selected category, integrating the instruction requested in the question. Its source citations, corpus/revision, and required placeholders appear directly before the code block. If the evidence cannot support such an example, the answer explains the missing information.

A rich example includes a concrete scenario, environment and initial values, expected outcome, and a final multi-statement code block with setup, the requested instruction, and follow-up. The shared RAG class validates structured generation and retries once if incomplete. If evidence cannot support the surrounding code, it explains the missing context. After updating the shared module, restart the kernel and rerun configuration and client setup to load the new implementation.

Each selected case reuses the **exact evidence from the preceding retrieval pass**
through the production answer method, avoiding a second embedding request or a
different retrieval result. It displays a retrieved-content summary and answer for
every case; generation failures retain the available summary. Completed answers
remain in `coverage_answer_results` if execution is interrupted.

The coding-answer validator may return `unsupported` for GUI or workflow
questions, which is separate from retrieval coverage. Review the explanation and
sources. A structurally `included` example still requires source checking.


In [ ]:
# | notest
coverage_answer_results, coverage_answer_errors = {}, {}
coverage_answer_metrics = await run_coverage_answers(
    app, coverage_cases, top_k=COVERAGE_TOP_K,
    retrieval_results=coverage_retrieval_results, retrieval_errors=coverage_retrieval_errors,
    results=coverage_answer_results, errors=coverage_answer_errors)
display(coverage_answer_metrics[
    ['case_id', 'category', 'hits', 'context_chars', 'omissions', 'missing_manuals', 'example_status', 'error']])


## General-operation manual version tests

This separate `general_operation` suite tests the two versions of
《新松机器人通用操作手册》:
[B-5](../assets/code-docs.4.0/06%20产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx)
and [C-1](../assets/code-docs.5.0/06%20产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx).

There are **38 prompts**: two different instruction questions for each of B-5's
eight subsections and C-1's six subsections in 9.1 (**28 questions**), plus
**10 alarm-code questions**, sampled reproducibly as five per manual from the
报警 tables, excluding 警告 and 提示信息. Original DOCX row locators, hashes,
reference excerpts and sampling seeds are preserved below. `profile` identifies
corpus 4.0 or 5.0, which differs from the manuals' V5.0 label.

Run application configuration, client setup and the shared coverage-helper cell
before this section; the preceding category retrieval/answer batches are optional.
Set `GENERAL_OPERATION_CASE_IDS` to a list of manual case IDs or leave it `None`
for all 38. This suite has its own top-k, selected cases, inspection ID and result
dictionaries, so it does not replace the category suite's results. No model request
occurs until its retrieval or answer cell is run. Run cleanup after both suites.


In [ ]:
# | notest
# Source locators are zero-based word/document.xml body blocks and table rows.
ALARM_SAMPLING = {'4.0': {'seed': 20261031,
         'population': 1229,
         'count': 5,
         'source_docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
         'sha256': '4e5f7dae024d703e27dd56a75c97606d0e9945e7c2206b148d492685469031db',
         'body_block': 3859,
         'rows': [150, 496, 627, 770, 1050]},
 '5.0': {'seed': 20261032,
         'population': 600,
         'count': 5,
         'source_docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
         'sha256': 'bfc19c6b11737c0a49c67565a4477d0d31e731fb26167a08b8e4800346aad286',
         'body_block': 3603,
         'rows': [85, 118, 154, 216, 580]}}

GENERAL_OPERATION_CASES = [{'id': 'general_40_9_1_1_single_output',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.1',
  'section_title': 'I/O类指令',
  'anchors': ['OUT'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用OUT OT#控制单个数字输出，说明IO号、ON/OFF含义，并给出先置位再复位的例子？',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3834,
                  'rows': [0, 1],
                  'excerpt': 'OUT OT# | 功能 | IO输出，单个IO\n | 格式 | OUT #<参数1>=<参数2>'}]},
 {'id': 'general_40_9_1_1_wait_input',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.1',
  'section_title': 'I/O类指令',
  'anchors': ['WAIT'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用WAIT等待输入信号满足条件，并设置等待时间？请说明输入号、状态值和超时参数。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3834,
                  'rows': [90, 91],
                  'excerpt': 'WAIT | 功能 | 等待条件满足。\n | 格式 | WAIT <参数1>=<参数2>  T=<参数3>'}]},
 {'id': 'general_40_9_1_2_goto_label',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.2',
  'section_title': '控制类1',
  'anchors': ['GOTO'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.2节“控制类1”回答：怎样用GOTO和标签L实现无条件跳转，说明被跳过的指令及跳转后的执行顺序？',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3836,
                  'rows': [0, 1],
                  'excerpt': 'GOTO | 功能 | 无条件跳转到标签处执行\n | 格式 | GOTO  L<参数项>'},
                 {'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3836,
                  'rows': [4, 5],
                  'excerpt': 'LABEL | 功能 | 标签，标明指令条件满足时，要跳转到的指令位置\n | 格式 | L<参数项>'}]},
 {'id': 'general_40_9_1_2_conditional_integer',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.2',
  'section_title': '控制类1',
  'anchors': ['IF'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.2节“控制类1”回答：怎样依据整型变量比较结果执行IF I条件转移？请解释比较参数和目标标签，并给出分支例子。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3836,
                  'rows': [23, 24],
                  'excerpt': 'IF I | 功能 | 如果整型变量满足比较条件，则作业跳转到标签处执行，否则继续执行IF指令的下一行指令。\n'
                             ' | 格式 | IF  <参数1><比较符><参数2>  L<参数3>'}]},
 {'id': 'general_40_9_1_3_subjob',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.3',
  'section_title': '控制类2',
  'anchors': ['CALL'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.3节“控制类2”回答：怎样用CALL调用子作业，并在子作业中用RET返回母作业？请区分无条件和输入IO条件调用。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3838,
                  'rows': [0, 1],
                  'excerpt': 'CALL | 功能 | 调用子作业\n | 格式 | CALL <调用条件><子作业名>'}]},
 {'id': 'general_40_9_1_3_position_shift',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.3',
  'section_title': '控制类2',
  'anchors': ['SHIFTON', 'SHIFTOFF'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.3节“控制类2”回答：怎样用SHIFTON按位置变量在指定坐标系内偏移运动，并用SHIFTOFF取消偏移？请解释F#的坐标系类型。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3838,
                  'rows': [4, 5],
                  'excerpt': 'SHIFT P | 功能 | 以设定的坐标系按位置变量中的值进行偏移\n | 格式 | SHIFTON  PR<参数1>  F#<参数2>'},
                 {'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3838,
                  'rows': [9, 10],
                  'excerpt': 'SHIFT OFF | 功能 | 停止所有偏移。\n | 格式 | SHIFTOFF'}]},
 {'id': 'general_40_9_1_4_linear_motion',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.4',
  'section_title': '运动类指令',
  'anchors': ['MOVL'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.4节“运动类指令”回答：怎样用MOVL使TCP以指定线速度运动到P1？请解释VL、ACC、CNT的含义和单位，并说明轨迹连续参数。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3840,
                  'rows': [7, 8],
                  'excerpt': 'MOVL P/PR | 功能 | 两点之间以直线插补方式进行运动。\n'
                             ' | 格式 | MOVL  P<参数1>  VL=<参数2> ACC=<参数3> CNT=<参数4>'}]},
 {'id': 'general_40_9_1_4_circular_motion',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.4',
  'section_title': '运动类指令',
  'anchors': ['MOVC'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.4节“运动类指令”回答：怎样用MOVC组织三点圆弧运动？请说明点位、VC、ACC、CNT参数以及圆弧示教要求。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3840,
                  'rows': [14, 15],
                  'excerpt': 'MOVC P/PR | 功能 | 三点之间以圆弧插补方式进行运动。\n'
                             ' | 格式 | MOVC P<参数1> VC=<参数2> ACC=<参数3> CNT=<参数4>'}]},
 {'id': 'general_40_9_1_5_user_frame',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.5',
  'section_title': '运算类指令',
  'anchors': ['SET UF'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.5节“运算类指令”回答：如何用SET UF设置用户坐标系号？请说明文件号范围及对自动运行和示教轴操作的影响。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3842,
                  'rows': [4, 5],
                  'excerpt': 'SET UF | 功能 | '
                             '设置用户坐标系号。该指令执行后，系统的当前用户坐标系号则被改变，用户坐标系号的改变不仅对自动执行的作业有影响，示教模式下的轴操作也使用的是新设定的用户坐标系。\n'
                             ' | 格式 | SET UF #<参数1>'}]},
 {'id': 'general_40_9_1_5_interrupt_position',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.5',
  'section_title': '运算类指令',
  'anchors': ['IRQP'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.5节“运算类指令”回答：如何在中断程序中获取中断发生瞬间的位置并存入PR10？请比较GET '
              'IRQP的规划值、实际值选择和中断等级要求。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3842,
                  'rows': [59, 60],
                  'excerpt': 'GET IRQP | 功能 | 获取中断发生瞬间的机器人当前位置的规划值或实际值赋值给位置\n'
                             '变量PR（需要在程序中做中断，然后在中断程序中使用此指令）\n'
                             ' | 格式 | IRQP<参数1> LEV=<参数2> P<参数3>'}]},
 {'id': 'general_40_9_1_6_integer_assignment',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.6',
  'section_title': '计算类指令',
  'anchors': ['SET'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.6节“计算类指令”回答：如何将整型变量I2的值赋给I1，再使用该值？请说明整型SET的参数类型和赋值方向。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3844,
                  'rows': [0, 1],
                  'excerpt': '整型 SET | 功能 | 赋值运算，将参数2的值赋给参数1\n | 格式 | SET  <参数1><参数2>'}]},
 {'id': 'general_40_9_1_6_byte_logic',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.6',
  'section_title': '计算类指令',
  'anchors': ['AND', 'XOR'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.6节“计算类指令”回答：字节变量的AND与XOR运算有什么区别？请说明结果保存到哪个参数，并分别给出有初始值的例子。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3850,
                  'rows': [5, 6],
                  'excerpt': '字节型\nAND | 功能 | 按位与运算，参数1与参数2按位求与，结果存在参数1中。\n | 格式 | AND  B<参数1>  B<参数2>'},
                 {'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3850,
                  'rows': [20, 21],
                  'excerpt': '字节型\nXOR | 功能 | 按位异或运算，参数1与参数2按位求异或，结果存在参数1中。\n | 格式 | XOR  B<参数1>  B<参数2>'}]},
 {'id': 'general_40_9_1_7_tcp_connect',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.7',
  'section_title': '通讯指令',
  'anchors': ['TCPCREATE'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.7节“通讯指令”回答：如何用TCPCREATE创建TCP连接？请解释NAME、IP、PORT、TIMEOUT、RET参数及如何判断连接结果。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3853,
                  'rows': [0, 1],
                  'excerpt': 'TCPCREATE  | 功能 | 创建一个 TCP 连接\n'
                             ' | 格式 | TCPCREATE NAME=<参数1> IP=<参数2> PORT=<参数3> TIMEOUT=<参数4> RET=<参数5>'}]},
 {'id': 'general_40_9_1_7_substring',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.7',
  'section_title': '通讯指令',
  'anchors': ['STRSUB'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.7节“通讯指令”回答：如何用STRSUB从字符串提取子串？请说明源字符串、起始位置、长度及目标字符串参数，并给出初始化和结果使用的例子。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3853,
                  'rows': [61, 62],
                  'excerpt': 'STRSUB | 功能 | 截取字符串的子串\n'
                             ' | 格式 | STRSUB STR=LS#<参数1> BEGIN=<参数2> END=<参数3> RET=<参数4>'}]},
 {'id': 'general_40_9_1_8_home_job',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.8',
  'section_title': '指令集',
  'anchors': ['A01_CMN HomePos'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.8节“指令集”回答：如何使用A01_CMN '
              'HomePos调用原位作业？请说明指令功能；若证据不足以组成完整返回原位程序，请明确缺少哪些条件。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3855,
                  'rows': [0, 1],
                  'excerpt': 'A01_CMN\nHomePos | 功能 | 调用指令集作业\n | 格式 | A01_CMN HomePos（原位作业）'}]},
 {'id': 'general_40_9_1_8_zone_request',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.8',
  'section_title': '指令集',
  'anchors': ['A01_CMN UserRequest'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.8节“指令集”回答：如何使用A01_CMN '
              'UserRequest申请和释放区域？请比较AllocateZone、ReleaseZone与Request、Finish，并说明UserNo参数和关联整型变量。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3855,
                  'rows': [2, 3],
                  'excerpt': 'A01_CMN\n'
                             'UserRequest | 功能 | 调用不同类型的指令集作业\n'
                             ' | 格式 | A01_CMN UserRequest<参数1> UserNo<参数2>'}]},
 {'id': 'general_50_9_1_1_joint_and_linear',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.1',
  'section_title': '运动类',
  'anchors': ['MOVJ', 'MOVL'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.1节“运动类”回答：MOVJ与MOVL分别采用什么插补方式，速度参数及单位有何不同？请用P[]格式说明两种运动的参数并组织一个例子。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3387,
                  'rows': [0, 1],
                  'excerpt': 'MOVJ | 功能 | 关节运动指令\n'
                             ' | 格式 | MOVJ P[值]\\PR[值] V = <参数2> ACC = <参数3> CNT = <参数4> \\ FINE'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3389,
                  'rows': [0, 1],
                  'excerpt': 'MOVL | 功能 | 直线运动指令\n'
                             ' | 格式 | MOVL P[值]\\PR[值] V = <参数2> ACC = <参数3> CNT = <参数4> \\ FINE'}]},
 {'id': 'general_50_9_1_1_circular_motion',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.1',
  'section_title': '运动类',
  'anchors': ['MOVC'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.1节“运动类”回答：如何用MOVC组织圆弧运动？请说明示教点、速度、加速度和轨迹连续参数，保持使用本版本的点位语法。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3391,
                  'rows': [0, 1],
                  'excerpt': 'MOVC | 功能 | 圆弧运动指令\n'
                             ' | 格式 | MOVL P[值]\\PR[值] V = <参数2> ACC = <参数3> CNT = <参数4> \\ FINE'}]},
 {'id': 'general_50_9_1_2_typed_assignment',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.2',
  'section_title': '基础指令',
  'anchors': ['SET'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.2节“基础指令”回答：如何用SET将I[2]赋给I[1]？请解释本版本支持的变量类型，并给出初始化、赋值和后续使用的例子。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3394,
                  'rows': [0, 1],
                  'excerpt': 'SET | 功能 | 赋值指令\n | 格式 | SET  <参数1>  <参数2>'}]},
 {'id': 'general_50_9_1_2_substring',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.2',
  'section_title': '基础指令',
  'anchors': ['STRSUB'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.2节“基础指令”回答：如何使用STRSUB从S[]字符串中提取一段子串？请说明参数含义、索引约定及结果保存位置。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3421,
                  'rows': [0, 1],
                  'excerpt': 'STRSUB | 功能 | 获取部分字符串，从参数1中获取参数2与参数3长度的字符串，结果放参数4中。\n'
                             ' | 格式 | STRSUB <参数1> <参数2> <参数3> <参数4>'}]},
 {'id': 'general_50_9_1_3_compound_expression',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.3',
  'section_title': '运算指令',
  'anchors': ['COP'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.3节“运算指令”回答：如何使用COP组合加减乘除运算，把计算结果写入整型或实型变量？请说明可用运算符及对P变量运算的限制。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3450,
                  'rows': [0, 1],
                  'excerpt': 'COP | 功能 | 运算类符合指令，支持各种运算的组合，组合没有长度限制。\n'
                             ' | 格式 | <参数1>  =  <参数2>  [运算符] <参数3> …'}]},
 {'id': 'general_50_9_1_3_square_root',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.3',
  'section_title': '运算指令',
  'anchors': ['SQRT'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.3节“运算指令”回答：如何使用SQRT计算平方根并保存结果？请说明输入、输出参数类型并给出初始化及后续使用的例子。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3464,
                  'rows': [0, 1],
                  'excerpt': 'SQRT | 功能 | 开平方指令，对参数2进行开方运算，结果存入参数1中。\n | 格式 | R[值] = SQRT <参数2>'}]},
 {'id': 'general_50_9_1_4_digital_io',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.4',
  'section_title': 'I/O指令',
  'anchors': ['IN', 'OUT'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.4节“I/O指令”回答：如何用IN读取数字输入、用OUT设置数字输出？请说明DI[]、DO[]及组IO参数的含义，保持使用本版本语法。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3488,
                  'rows': [0, 1],
                  'excerpt': 'IN | 功能 | 将I/O赋给变量I[值]\n | 格式 | IN I[值] = <参数2> '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3491,
                  'rows': [0, 1],
                  'excerpt': 'OUT | 功能 | 对I/O和组I/O进行输出。\n | 格式 | OUT <参数1> = <参数2> '}]},
 {'id': 'general_50_9_1_4_analog_io',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.4',
  'section_title': 'I/O指令',
  'anchors': ['ANIN', 'ANOUT'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.4节“I/O指令”回答：如何用ANIN读取模拟量并用ANOUT输出模拟量？请说明通道、数值和变量类型参数。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3497,
                  'rows': [0, 1],
                  'excerpt': 'ANIN | 功能 | 模拟量输出信号，是连续的信号，一般表示电压、电流等数据。将模拟量输入I/O赋值给I变量。\n'
                             ' | 格式 | ANIN <参数1> = <参数2> '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3500,
                  'rows': [0, 1],
                  'excerpt': 'ANOUT | 功能 | 模拟量输出信号，是连续的信号，一般表示电压、电流等数据。将变量值转换成模拟量输出到指定的I/0中。\n'
                             ' | 格式 | ANOUT <参数1> = <参数2> '}]},
 {'id': 'general_50_9_1_5_conditional_branch',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.5',
  'section_title': '控制类',
  'anchors': ['IF', 'ELSEIF', 'ELSE', 'ENDIF'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.5节“控制类”回答：如何用IF、ELSEIF、ELSE、ENDIF组织多分支判断？请说明比较运算符及AND、OR的作用，并给出变量条件示例。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3502,
                  'rows': [0, 1],
                  'excerpt': 'IF OP | 功能 | 条件判断指令，对条件进行判断，并根据结果进行后续指令的执行。\n'
                             ' | 格式 | IF <参数1> 操作符 <参数2> <附加指令>'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3507,
                  'rows': [0, 1],
                  'excerpt': 'ELSEIF OP | 功能 | 满足条件执行区间内指令。\n'
                             ' | 格式 | ELSEIF <参数1> 操作符 <参数2> AND <参数1> 操作符 <参数2> <附加指令>'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3510,
                  'rows': [0, 1],
                  'excerpt': 'ELSE | 功能 | 与IF指令一起使用，IF条件未满足，执行区间内指令。\n | 格式 | ELSE'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3512,
                  'rows': [0, 1],
                  'excerpt': 'ENDIF | 功能 | 与IF指令一起使用，条件跳转指令结束。\n | 格式 | ENDIF'}]},
 {'id': 'general_50_9_1_5_for_loop',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.5',
  'section_title': '控制类',
  'anchors': ['FOR', 'ENDFOR', 'BREAK', 'CONTINUE'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.5节“控制类”回答：如何用FOR和ENDFOR组织带STEP的计数循环？请说明循环变量类型、条件和步长，并区分BREAK与CONTINUE。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3556,
                  'rows': [0, 1],
                  'excerpt': 'FOR...ENDFOR | 功能 | FOR循环指令。\n | 格式 | FOR<参数1>=<参数2><参数3><参数4><参数5><参数6>'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3561,
                  'rows': [0, 1],
                  'excerpt': 'BREAK | 功能 | BREAK语句。\n | 格式 | BREAK '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3563,
                  'rows': [0, 1],
                  'excerpt': 'CONTINUE | 功能 | CONTINUE语句。\n | 格式 | CONTINUE '}]},
 {'id': 'general_50_9_1_6_interrupt_enable',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.6',
  'section_title': '操作类',
  'anchors': ['IRQON LEV', 'IRQOFF'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.6节“操作类”回答：如何使用IRQON LEV启用输入信号触发的中断并用IRQOFF关闭？请说明等级、比较条件和作业名参数。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3568,
                  'rows': [0, 1],
                  'excerpt': 'IRQON LEV | 功能 | 中断开启指令。\n | 格式 | IRQON LEV = <参数1> <参数2> [运算符] <参数3> [作业名]  '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3571,
                  'rows': [0, 1],
                  'excerpt': 'IRQOFF | 功能 | 中断结束指令。\n | 格式 | IRQOFF LEV = <参数1> '}]},
 {'id': 'general_50_9_1_6_tcp_receive',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.6',
  'section_title': '操作类',
  'anchors': ['TCPRECV'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.6节“操作类”回答：如何使用TCPRECV接收服务端字符串？请说明连接名称、目标S[]变量、最大长度、TIMEOUT及接收结果参数。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3586,
                  'rows': [0, 1],
                  'excerpt': 'TCPRECV | 功能 | TCP接收指令并，接受服务端字符串数据，并存储至参数2中。\n'
                             ' | 格式 | TCPRECV <参数1> <参数2> <参数3> TIMEOUT = <参数4> <参数5>'}]},
 {'id': 'general_40_alarm_1253',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1253',
  'anchors': ['1253'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码1253的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [150],
                  'excerpt': '1253 | PROT0连续接收到无效帧错误:SV   | 山洋驱动器报警，[参数]为报警驱动器对应轴号 | '
                             '[清除报警]可复位，对应驱动器报警代码10，反复出现请联系新松售后人员'}],
  'reference_alarm': {'message': 'PROT0连续接收到无效帧错误:SV  ',
                      'cause': '山洋驱动器报警，[参数]为报警驱动器对应轴号',
                      'remedy': '[清除报警]可复位，对应驱动器报警代码10，反复出现请联系新松售后人员'}},
 {'id': 'general_40_alarm_1613',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1613',
  'anchors': ['1613'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码1613的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [496],
                  'excerpt': '1613 | 限位开关异常:SV  | 电机正向运行时，检测到负限位开关触发\n'
                             '电机反向运行时，检测到正限位开关触发 | 检测接线和参数设置，确保限位开关的极性定义与接线正确'}],
  'reference_alarm': {'message': '限位开关异常:SV ',
                      'cause': '电机正向运行时，检测到负限位开关触发\n电机反向运行时，检测到正限位开关触发',
                      'remedy': '检测接线和参数设置，确保限位开关的极性定义与接线正确'}},
 {'id': 'general_40_alarm_1744',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1744',
  'anchors': ['1744'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码1744的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [627],
                  'excerpt': '1744 | 运行统计信息初始化失败:SV | 初始化失败 | 请进行复位，或重启系统，反复出现请联系新松售后人员'}],
  'reference_alarm': {'message': '运行统计信息初始化失败:SV', 'cause': '初始化失败', 'remedy': '请进行复位，或重启系统，反复出现请联系新松售后人员'}},
 {'id': 'general_40_alarm_8150',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '8150',
  'anchors': ['8150'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码8150的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [770],
                  'excerpt': '8150 | 绝对位置遗失:SV  | 条件：绝对型编码器因为电池低电压或供电中断而遗失内部所记录的圈数。原因：1. 电池电压过低。2. '
                             '在驱动器控制电源Off 的状况下更换电池。3. 启动绝对型功能后，未安装电池。4. 电池供电线路接触不良或断线。 | 1. '
                             '检查电池电压是否低于2.9V；更换电池后，重新建立绝对型原点坐标。2. 请勿在驱动器控制电源Off 的状况下更换或移除电池。3. 建议检查事项如下：(a) '
                             '安装电池。(b) 检查电池外接盒跟驱动器的电池电源接线。(c) 检查编码器配线。4. 确认接线是否可让电池电力正常供给编码器，重新建立绝对型原点坐标。'}],
  'reference_alarm': {'message': '绝对位置遗失:SV ',
                      'cause': '条件：绝对型编码器因为电池低电压或供电中断而遗失内部所记录的圈数。原因：1. 电池电压过低。2. 在驱动器控制电源Off 的状况下更换电池。3. '
                               '启动绝对型功能后，未安装电池。4. 电池供电线路接触不良或断线。',
                      'remedy': '1. 检查电池电压是否低于2.9V；更换电池后，重新建立绝对型原点坐标。2. 请勿在驱动器控制电源Off 的状况下更换或移除电池。3. '
                                '建议检查事项如下：(a) 安装电池。(b) 检查电池外接盒跟驱动器的电池电源接线。(c) 检查编码器配线。4. '
                                '确认接线是否可让电池电力正常供给编码器，重新建立绝对型原点坐标。'}},
 {'id': 'general_40_alarm_8438',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '8438',
  'anchors': ['8438'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码8438的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [1050],
                  'excerpt': '8438 | 输出对地短路:SV | 1. 驱动器动力线缆(U V W)对地发生短路2. 电机对地短路3. 驱动器故障 | '
                             '1、拔掉电机线缆，分别测量驱动器动力线缆U V W是否对地(PE) '
                             '短路。重新接线或更换驱动器动力线缆。2、确保驱动器动力线缆、电机线缆紧固连接后，分别测量驱动器U V W 端与接地线(PE) '
                             '之间的绝缘电阻是否为兆欧姆(MΩ) 级数值。更换电机。3、将驱动器动力线缆从伺服驱动器上卸下，多次接通电源后仍报故障。更换伺服驱动器。'}],
  'reference_alarm': {'message': '输出对地短路:SV',
                      'cause': '1. 驱动器动力线缆(U V W)对地发生短路2. 电机对地短路3. 驱动器故障',
                      'remedy': '1、拔掉电机线缆，分别测量驱动器动力线缆U V W是否对地(PE) '
                                '短路。重新接线或更换驱动器动力线缆。2、确保驱动器动力线缆、电机线缆紧固连接后，分别测量驱动器U V W 端与接地线(PE) '
                                '之间的绝缘电阻是否为兆欧姆(MΩ) 级数值。更换电机。3、将驱动器动力线缆从伺服驱动器上卸下，多次接通电源后仍报故障。更换伺服驱动器。'}},
 {'id': 'general_50_alarm_1372',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1372',
  'anchors': ['1372'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1372的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [85],
                  'excerpt': '1372 | CPU过载:SV   | 1.控制指令超过CPU负载能力2.驱动器损坏。 | 处理方法：降低控制指令操作频率；更换或维修驱动器'}],
  'reference_alarm': {'message': 'CPU过载:SV  ',
                      'cause': '1.控制指令超过CPU负载能力2.驱动器损坏。',
                      'remedy': '处理方法：降低控制指令操作频率；更换或维修驱动器'}},
 {'id': 'general_50_alarm_1405',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1405',
  'anchors': ['1405'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1405的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [118],
                  'excerpt': '1405 | 位置硬超限:SV   | 驱动器外接的位置限位开关触发； | 处理方法：检查位置指令规划范围'}],
  'reference_alarm': {'message': '位置硬超限:SV  ', 'cause': '驱动器外接的位置限位开关触发；', 'remedy': '处理方法：检查位置指令规划范围'}},
 {'id': 'general_50_alarm_1507',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1507',
  'anchors': ['1507'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1507的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [154],
                  'excerpt': '1507 | 轴内部数据记录初始化:SV | 对应轴号初始化 | 处理方法：联系技术人员。'}],
  'reference_alarm': {'message': '轴内部数据记录初始化:SV', 'cause': '对应轴号初始化', 'remedy': '处理方法：联系技术人员。'}},
 {'id': 'general_50_alarm_1569',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1569',
  'anchors': ['1569'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1569的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [216],
                  'excerpt': '1569 | 伺服参数内部转换初始化:SV  | 对应轴号初始化 | 处理方法：联系技术人员。'}],
  'reference_alarm': {'message': '伺服参数内部转换初始化:SV ', 'cause': '对应轴号初始化', 'remedy': '处理方法：联系技术人员。'}},
 {'id': 'general_50_alarm_6035',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '6035',
  'anchors': ['6035'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码6035的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [580],
                  'excerpt': '6035 | 运动队列运动属运动类别错误 |  | '}],
  'reference_alarm': {'message': '运动队列运动属运动类别错误', 'cause': '', 'remedy': ''}}]


In [ ]:
# | notest
GENERAL_OPERATION_TOP_K = 8
GENERAL_OPERATION_CASE_IDS = None  # Example: ['general_40_9_1_6_integer_assignment', 'general_50_alarm_6035']
known_general_operation_ids = {case['id'] for case in GENERAL_OPERATION_CASES}
assert {case['category'] for case in GENERAL_OPERATION_CASES} == {'general_operation'}
if GENERAL_OPERATION_CASE_IDS is not None:
    if not GENERAL_OPERATION_CASE_IDS or set(GENERAL_OPERATION_CASE_IDS) - known_general_operation_ids:
        raise ValueError('Select at least one known general-operation case ID')
general_operation_cases = [case for case in GENERAL_OPERATION_CASES
                          if GENERAL_OPERATION_CASE_IDS is None or case['id'] in GENERAL_OPERATION_CASE_IDS]
GENERAL_OPERATION_INSPECT_CASE_ID = general_operation_cases[0]['id']
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(pd.DataFrame(general_operation_cases)[
        ['id', 'kind', 'profile', 'revision', 'section', 'section_title', 'question']])
display(app.snapshot('general_operation')[2])


### Retrieve and inspect manual evidence

Retrieve the selected version-specific questions and report expected manual,
subsection and command/alarm anchors separately from the returned hit count.
Production retrieval still searches both mapped revisions; the diagnostics check
whether the intended revision was found. These checks are not answer correctness
scores. The inspection call takes this suite's cases and result dictionaries
explicitly; change `GENERAL_OPERATION_INSPECT_CASE_ID` to inspect another case.


In [ ]:
# | notest
general_operation_retrieval_results, general_operation_retrieval_errors = {}, {}
general_operation_retrieval_metrics = await run_coverage_retrievals(
    app, general_operation_cases, top_k=GENERAL_OPERATION_TOP_K,
    results=general_operation_retrieval_results, errors=general_operation_retrieval_errors)
display(general_operation_retrieval_metrics)
show_coverage_retrieval(
    GENERAL_OPERATION_INSPECT_CASE_ID, cases=general_operation_cases,
    results=general_operation_retrieval_results, errors=general_operation_retrieval_errors)


### Retrieved summaries and manual answers

Answers use the exact evidence from this section's retrieval pass through the same
production answer method as the category suite. Completed results and available
summaries remain inspectable after failure or interruption. Reference excerpts are
reviewer data and are never appended to model context.

Alarm questions ask for meaning, causes and remedies without invented programs.
Empty cause/remedy fields remain explicitly unknown. The coding-answer validator
may return `unsupported` for a diagnostic question; review the explanation and
references independently of that status and the retrieval coverage checks.


In [ ]:
# | notest
general_operation_answer_results, general_operation_answer_errors = {}, {}
general_operation_answer_metrics = await run_coverage_answers(
    app, general_operation_cases, top_k=GENERAL_OPERATION_TOP_K,
    retrieval_results=general_operation_retrieval_results,
    retrieval_errors=general_operation_retrieval_errors,
    results=general_operation_answer_results, errors=general_operation_answer_errors)
display(general_operation_answer_metrics)


## Cleanup
Run after all selected category and manual tests, or after interruption.


In [ ]:
# | notest
try:
    if 'app' in globals(): app.close()
finally:
    for client_name in ('embedding_client', 'generation_client'):
        if client_name in globals(): await globals()[client_name]._client.aclose()